# Auditoría de datos


## Paso 1: Reconocimiento inicial

In [99]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from itertools import combinations, permutations


- El número de columnas nos puede dar una idea de cuántas entidades potenciales hay.

In [100]:
PATH = "moto_results.csv"

df = pd.read_csv(PATH) 

print("=== DIMENSIONES ===")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

print("\n=== COLUMNAS ===")
print(list(df.columns))


print("\n=== NOMBRES Y TIPOS DE COLUMNAS ===")
print(df.dtypes)

print("\n=== PRIMERAS 5 FILAS ===")
print(df.head())

=== DIMENSIONES ===
Filas: 29931, Columnas: 17

=== COLUMNAS ===
['year', 'sequence', 'category', 'rider_first_name', 'rider_last_name', 'rider_number', 'rider_country', 'team_name', 'bike', 'position', 'points', 'speed', 'time', 'race_name', 'circuit_name', 'circuit_country', 'date']

=== NOMBRES Y TIPOS DE COLUMNAS ===
year                  int64
sequence              int64
category             object
rider_first_name     object
rider_last_name      object
rider_number        float64
rider_country        object
team_name            object
bike                 object
position              int64
points                int64
speed               float64
time                 object
race_name            object
circuit_name         object
circuit_country      object
date                 object
dtype: object

=== PRIMERAS 5 FILAS ===
   year  sequence category rider_first_name rider_last_name  rider_number  \
0  2021        16    Moto2           Fermín        Aldeguer          54.0   
1  2021

In [101]:
df = df.convert_dtypes()
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
df.dtypes


year                         Int64
sequence                     Int64
category            string[python]
rider_first_name    string[python]
rider_last_name     string[python]
rider_number                 Int64
rider_country       string[python]
team_name           string[python]
bike                string[python]
position                     Int64
points                       Int64
speed                      Float64
time                string[python]
race_name           string[python]
circuit_name        string[python]
circuit_country     string[python]
date                datetime64[ns]
dtype: object

In [102]:
print("=== ESTADÍSTICAS BÁSICAS DEL DATASET ===\n")
print(f"Rango de temporadas: {df["year"].min()} - {df["year"].max()}\n")

print(f"Categorías: {df["category"].nunique()}")
print(df["category"].value_counts().to_string())

print(f"\nPosiciones: ")
# print(df["position"].value_counts(ascending=True).to_string())
print(f"Hay {(df["position"]<0).sum()} valores negativos en la columna Position, lo que puede representar situaciones como DNF, descalificaciones, etc")

print(f"\nEquipos: {df["team_name"].nunique()}")
print(df["team_name"].value_counts())
# Observamos que el equipo "?" tiene exactamente el mismo número de valores que valores nulos en rider_number.

=== ESTADÍSTICAS BÁSICAS DEL DATASET ===

Rango de temporadas: 2000 - 2021

Categorías: 7
category
MotoGP    7023
Moto2     6755
125cc     6013
Moto3     5518
250cc     3856
500cc      502
MotoE      264

Posiciones: 
Hay 5475 valores negativos en la columna Position, lo que puede representar situaciones como DNF, descalificaciones, etc

Equipos: 970
team_name
?                          5127
Repsol Honda Team           598
Red Bull KTM Ajo            534
SKY Racing Team VR46        399
Ducati Team                 393
                           ... 
NTS T.Pro Project             1
Pons Racing Junior Team       1
FAB-Racing                    1
Abbink GP                     1
HP Moto Kalex                 1
Name: count, Length: 970, dtype: Int64


In [103]:
heat = df.groupby(['year','category']).size().reset_index(name='count')
heat_pivot = heat.pivot(index='category', columns='year', values='count').fillna(0)

fig = go.Figure(go.Heatmap(
    z=heat_pivot.values,
    x=heat_pivot.columns.astype(str),
    y=heat_pivot.index,
    colorscale='Teal',
    text=heat_pivot.values.astype(int),
    texttemplate='%{text}',
    textfont={"size": 9}
))
fig.update_xaxes(title_text='Año')
fig.update_yaxes(title_text='Categoría')
fig.update_layout(title='Registros por categoría y temporada')
fig.show()

In [104]:
# Cambiar el país del circuito MotorLand Aragón a ES
df.loc[df["circuit_name"] == "MotorLand Aragón", "circuit_country"] = "ES"

# Verificar que el cambio se ha aplicado correctamente
df[df["circuit_name"] == "MotorLand Aragón"]["circuit_country"].unique()

<StringArray>
['ES']
Length: 1, dtype: string

In [105]:
# Al realizar las consultas, nos damos cuenta que debemos normalizar los nombres de circuito: strip + title case 
df['circuit_name'] = (df['circuit_name']
                      .str.strip()           
                      .str.title())         

### Valores nulos

In [106]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29931 entries, 0 to 29930
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   year              29931 non-null  Int64         
 1   sequence          29931 non-null  Int64         
 2   category          29931 non-null  string        
 3   rider_first_name  29931 non-null  string        
 4   rider_last_name   29931 non-null  string        
 5   rider_number      24804 non-null  Int64         
 6   rider_country     29931 non-null  string        
 7   team_name         29931 non-null  string        
 8   bike              29931 non-null  string        
 9   position          29931 non-null  Int64         
 10  points            29931 non-null  Int64         
 11  speed             28939 non-null  Float64       
 12  time              29930 non-null  string        
 13  race_name         29931 non-null  string        
 14  circuit_name      2993

In [107]:
nulls = df.isnull().sum().reset_index()
nulls.columns = ['columna', 'nulos']
nulls = nulls[nulls['nulos'] > 0].sort_values('nulos', ascending=True)

fig = go.Figure(go.Bar(x=nulls['nulos'], y=nulls['columna'], orientation='h'))
fig.update_xaxes(title_text='Nº valores nulos')
fig.update_yaxes(title_text='Columna')
fig.update_layout(title='Valores nulos por columna')
fig.show()

In [108]:
print(f"La columna rider_number tiene {sum(df["rider_number"].isnull())} valores nulos")
# Muchos pilotos (categorias inferiores como 125cc, 250cc, 500cc) no tienen número registrado

La columna rider_number tiene 5127 valores nulos


In [109]:
# ¿Son exactamente las mismas filas?
mask_team = df['team_name'] == '?'
mask_number = df['rider_number'].isnull()
print(f"Filas con team='?': {mask_team.sum()}")
print(f"Filas con rider_number nulo: {mask_number.sum()}")
print(f"¿Son las mismas filas? {(mask_team == mask_number).all()}")

Filas con team='?': 5127
Filas con rider_number nulo: 5127
¿Son las mismas filas? True


In [110]:
print(f"La columna speed tiene {sum(df["speed"].isnull())} valores nulos")
# Posibles pilotos descalificados antes de comenzar

La columna speed tiene 992 valores nulos


In [111]:
unknown = df[df['team_name'] == '?'].groupby('year').size().reset_index(name='count')

fig = px.bar(unknown, x='year', y='count',
             title='Registros sin equipo identificado ("?") por año')
fig.update_xaxes(title_text='Año', dtick=1)
fig.update_yaxes(title_text='Nº registros')
fig.show()

En este primer paso podemos anticipar al menos 5 entidades candidatas:

```plaintext
PILOTO       → rider_first_name, rider_last_name, rider_country
EQUIPO       → team_name
MOTO         → bike
CIRCUITO     → circuit_name, circuit_country
CARRERA      → race_name, circuit_name, year, sequence, category, date
RESULTADO    → (relación entre PILOTO, CARRERA, EQUIPO) + position, points, speed, time
```

## Paso 2: Dependencias Funcionales

En este paso vamos a analizar qué columna identifica de forma única a cada entidad y descubrir qué atributos pertenecen a qué tabla.

In [112]:
# ¿Un piloto puede estar en varios equipos el mismo año y categoría?
piloto_equipo = df[df['team_name'] != '?'].groupby(
    ['year', 'category', 'rider_first_name', 'rider_last_name']
)['team_name'].nunique()
multi_equipo = piloto_equipo[piloto_equipo > 1]
print(f"Casos encontrados: {len(multi_equipo)}")

Casos encontrados: 329


In [113]:
# 1. ¿El número de dorsal identifica al piloto dentro de año y categoría?
check = df[df["rider_number"].notna()].groupby(["year", "category", "rider_number"])[["rider_first_name", "rider_last_name"]].nunique()

coincidencias = check[(check["rider_first_name"] > 1) | (check["rider_last_name"] > 1)]
print(f"Coincidencias encontradas: {len(coincidencias)}\n")

if len(coincidencias) > 0:
    df["rider_full_name"] = df["rider_first_name"] + " " + df["rider_last_name"]
    nombres_por_dorsal = df[df["rider_number"].notna()].groupby(["year", "category", "rider_number"])["rider_full_name"].unique()
    coincidencias_pilotos = nombres_por_dorsal[nombres_por_dorsal.apply(len) > 1]
    for idx, pilotos in coincidencias_pilotos.head().items():
        año, categoria, dorsal = idx
        print(f"Año: {año} | Cat: {categoria} | Dorsal: {dorsal} -> Pilotos: {', '.join(pilotos)}")

    

Coincidencias encontradas: 14

Año: 2006 | Cat: 125cc | Dorsal: 80 -> Pilotos: Doni Tata Pradita, Tito Rabat
Año: 2006 | Cat: MotoGP | Dorsal: 8 -> Pilotos: Garry Mccoy, Naoki Matsudo
Año: 2007 | Cat: 125cc | Dorsal: 51 -> Pilotos: Stevie Bonsey, Steve Bonsey
Año: 2009 | Cat: 125cc | Dorsal: 76 -> Pilotos: Ivan Maestro, Toni Finsterbusch
Año: 2010 | Cat: 125cc | Dorsal: 57 -> Pilotos: Joel Taylor, Isaac Viñales


In [114]:
# 2. ¿Un piloto siempre tiene el mismo país registrado?
pais_piloto = df.groupby(["rider_first_name", "rider_last_name"])["rider_country"].nunique()

multiple_pais_piloto = pais_piloto[pais_piloto > 1]

print(f"Pilotos con más de un país registrado: {len(multiple_pais_piloto)}")


Pilotos con más de un país registrado: 0


In [115]:
# 3. ¿Un circuito siempre pertenece al mismo país?
pais_circuito = df.groupby("circuit_name")["circuit_country"].nunique()

coincidencias_circuito = pais_circuito[pais_circuito > 1]
print(f"Circuitos con más de un país: {len(coincidencias_circuito)}")

Circuitos con más de un país: 0


In [116]:
# 4. ¿El nombre de la carrera siempre apunta al mismo circuito?
carrera_circuito = df.groupby("race_name")["circuit_name"].nunique()
coincidencias_carrera = carrera_circuito[carrera_circuito > 1]

print(f"Carreras con más de un circuito: {len(coincidencias_carrera)}")

if len(coincidencias_carrera) > 0:
    print(df[df["race_name"].isin(coincidencias_carrera.index)].groupby("race_name")["circuit_name"].unique())

Carreras con más de un circuito: 1
race_name
Japanese Grand Prix    [Twin Ring Motegi, Suzuka Circuit]
Name: circuit_name, dtype: object


In [117]:
# 5. ¿(year, sequence, category) identifica una única carrera?
carrera_unica = df.groupby(["year", "sequence", "category"])["race_name"].nunique()

coincidencias_carrera = carrera_unica[carrera_unica > 1]
print(f"Combinaciones con más de un race_name: {len(coincidencias_carrera)}")

Combinaciones con más de un race_name: 0


In [118]:
# 6. ¿Un equipo siempre usa la misma moto?
equipo_moto = df.groupby("team_name")["bike"].unique()
multiple_moto = equipo_moto[equipo_moto.apply(len) > 1]
print(f"Equipos con más de un tipo de moto: {len(multiple_moto)}\n")

if len(multiple_moto) > 0:
    for equipo, moto in multiple_moto.head().items():
        motos_validas = [str(i) for i in moto if pd.notna(i)]
        print(f"Equipo: {equipo} -> Motos: {', '.join(motos_validas)}")


Equipos con más de un tipo de moto: 116

Equipo: +EGO Speed Up -> Motos: Boscoscuro, Speed Up
Equipo: ? -> Motos: Aprilia, Malaguti, KTM, Derbi, Honda, Gilera, Yamaha, Kawasaki, Ducati, Blata, Suzuki, Proton KR, Harris WCM, Moriwaki, Sabre V4, MD211VF PROTO, Italjet, TSR, Erp Honda, Pulse, Paton, NER Honda, Modenas KR3
Equipo: AGR Team -> Motos: Kalex, KTM
Equipo: Abbink Bos Racing -> Motos: Seel, Honda
Equipo: Aeroport de Castello - Ajo -> Motos: Aprilia, FTR


### Conclusiones obtenidas tras el análisis:

- **La clave natural del Piloto no existe en la tabla**: ni el número ni el nombre sirven solos. Es necesario generar un ``id_piloto`` artificial.

- **La clave natural de Carrera es ``(year, sequence, category)``**: es la única combinación que identifica una carrera sin ambigüedad.

- **``Bike`` no pertenece a Equipo**: pertenece al resultado de cada piloto en cada carrera, ya que el equipo puede usar varias marcas.

- **``Race_name`` no es fiable como identificador**: el mismo nombre puede referirse a una carrera en circuitos distintos.

```plaintext
PILOTO     (id_piloto PK, rider_first_name, rider_last_name, rider_country)
CIRCUITO   (id_circuito PK, circuit_name, circuit_country)  
CARRERA    (year PK, sequence PK, category PK, race_name, date, id_circuito FK)
EQUIPO     (id_equipo PK, team_name)
RESULTADO  (id_piloto FK, year FK, sequence FK, category FK, id_equipo FK, bike, position, points, speed, time)
```

## Paso 3: Análisis de valores especiales en ``position``

In [119]:
print(f"El {(df["position"]<=0).sum() / len(df) * 100:.2f}% de todas las filas tienen una posición negativa o cero")

El 18.29% de todas las filas tienen una posición negativa o cero


In [120]:
print(df['position'].value_counts().sort_index().head(10))

position
-5      60
-4     552
-3      31
-2     121
-1    4711
1     1121
2     1121
3     1121
4     1121
5     1121
Name: count, dtype: Int64


In [121]:
resumen = df[df['position'] <= 0].groupby('position').agg(
    total            = ('position', 'count'),
    puntos_siempre_0 = ('points',   lambda x: (x == 0).all()),
    speed_nulos      = ('speed',    lambda x: x.isna().sum()),
    ejemplo_time     = ('time',     lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else 'N/A')
)
print(resumen)

          total  puntos_siempre_0  speed_nulos ejemplo_time
position                                                   
-5           60                 1         36.0        0 Lap
-4          552                 1        552.0        0 Lap
-3           31                 1         31.0        0 Lap
-2          121                 1        121.0        0 Lap
-1         4711                 1        227.0       9 Laps


In [122]:
# Verificación crítica: ¿algún valor negativo acumula puntos?
assert (df[df['position'] <= 0]['points'] == 0).all(), "Hay posiciones negativas con puntos!"
print("Confirmado: todos los valores negativos tienen points = 0")
print("→ SUM(points) es seguro")

Confirmado: todos los valores negativos tienen points = 0
→ SUM(points) es seguro


In [123]:
print("\n=== EJEMPLOS REALES ===")
for val in sorted(df['position'].unique()):
    if val <= 0:
        sample = df[df['position'] == val][
            ['rider_first_name', 'rider_last_name', 'time', 'speed', 'points']
        ].head(2)
        print(f"\n  position = {val}:")
        print(sample.to_string(index=False))


=== EJEMPLOS REALES ===

  position = -5:
rider_first_name rider_last_name   time  speed  points
         Lorenzo     Dalla Porta  0 Lap   <NA>       0
           Kaito            Toba 6 Laps  106.7       0

  position = -4:
rider_first_name rider_last_name  time  speed  points
           Jorge         Navarro 0 Lap   <NA>       0
            Tony        Arbolino 0 Lap   <NA>       0

  position = -3:
rider_first_name rider_last_name  time  speed  points
             Sam           Lowes 0 Lap   <NA>       0
           Karel         Abraham 0 Lap   <NA>       0

  position = -2:
rider_first_name rider_last_name  time  speed  points
          Carlos           Tatay 0 Lap   <NA>       0
        Riccardo           Rossi 0 Lap   <NA>       0

  position = -1:
rider_first_name rider_last_name    time  speed  points
            Tony        Arbolino  9 Laps  154.5       0
         Cameron        Beaubier 17 Laps  154.2       0


In [124]:
neg = df[df['position'] <= 0].copy()
neg['tipo'] = neg['position'].map({-1:'DNF', -2:'DNS', -3:'DSQ', -4:'NC', -5:'Otro'})
v1 = neg.groupby(['category','tipo']).size().reset_index(name='count')

fig = px.bar(v1, x='category', y='count', color='tipo', barmode='group',
             title='Posiciones especiales por categoría')
fig.update_xaxes(title_text='Categoría')
fig.update_yaxes(title_text='Nº registros')
fig.show()


> "Los valores negativos en la columna position representan situaciones de no clasificación. El valor -1, acompañado de un tiempo en vueltas, corresponde a abandonos durante la carrera (DNF). Los valores -2 a -5, con speed = NULL y time = "0 Lap", representan distintos grados de no participación (DNS, DSQ o exclusión). En todos los casos, points = 0, por lo que no afectan al cómputo del campeonato."

# Normalización


### Primera Forma Normal (1FN)

Una tabla está en 1FN si todas sus columnas contienen **valores atómicos**, sin valores múltiples ni repetidos. 

Por tanto lo que vamos a hacer para probar esta propiedad es ver si hay columnas con listas o estructuras complejas.

In [125]:
# Comprobamos si hay columnas que contengan estructuras complejas (listas, diccionarios, tuplas)
no_atomicos = df.map(lambda x: isinstance(x, (list, dict, tuple))).sum()

print("=== Comprobación de Primera Forma Normal (1FN) ===\n")

print(no_atomicos)
print()
if no_atomicos.sum() == 0:
    print("- Todas las columnas contienen valores atómicos.")
    print("- No se detectaron listas, diccionarios ni tuplas anidadas en ninguna columna.")
    print("- El dataset cumple con la Primera Forma Normal (1FN).")
else:
    print("- Se han encontrado valores no atómicos en las siguientes columnas:")
    print(no_atomicos[no_atomicos > 0])

=== Comprobación de Primera Forma Normal (1FN) ===

year                0
sequence            0
category            0
rider_first_name    0
rider_last_name     0
rider_number        0
rider_country       0
team_name           0
bike                0
position            0
points              0
speed               0
time                0
race_name           0
circuit_name        0
circuit_country     0
date                0
rider_full_name     0
dtype: int64

- Todas las columnas contienen valores atómicos.
- No se detectaron listas, diccionarios ni tuplas anidadas en ninguna columna.
- El dataset cumple con la Primera Forma Normal (1FN).


### Segunda Forma Normal (2FN)

**Condición**: estar en 1FN + todos los atributos no clave dependen de toda la clave primaria, no solo de una parte. Solo afecta a tablas con **clave compuesta**.

Como ya hemos demostrado que nuestro modelo cumple con la 1FN, nos queda analizar las dos tablas que tienen claves compuestas: **``Carrera``** y **``Resultado``**

#### ``CARRERA``

La clave compuesta es ``(año, secuencia, categoria)``. Nos planteamos la siguiente pregunta:

- **¿Cada atributo depende de la clave completa?**

##### 1. Comprobar la Dependencia Completa

Primero verificamos que la clave compuesta determina unívocamente a los atributos de CARRERA:

In [126]:
# Clave compuesta de CARRERA
clave_carrera = ["year", "sequence", "category"]

# Atributos propios de CARRERA 
atributos_carrera = ["race_name", "date"]

print("=== Comprobación de Dependencia Funcional ===\n")
for attr in atributos_carrera:
    # Agrupamos por la clave compuesta y contamos los valores únicos del atributo
    max_valores = df.groupby(clave_carrera)[attr].nunique().max()
    
    if max_valores == 1:
        print(f"- {attr}: Depende funcionalmente de la clave (máx {max_valores} valor por grupo).")
    else:
        print(f"- {attr}: NO depende funcionalmente (hasta {max_valores} valores distintos por grupo).")

=== Comprobación de Dependencia Funcional ===

- race_name: Depende funcionalmente de la clave (máx 1 valor por grupo).
- date: Depende funcionalmente de la clave (máx 1 valor por grupo).


##### 2. Comprobar Dependencias Parciales

Para que esté en 2FN, los atributos no solo deben depender de la clave completa, sino que no deben depender de un subconjunto de ella.

In [127]:
# Clave compuesta de CARRERA
clave_carrera = ["year", "sequence", "category"]

# Atributos propios de CARRERA 
atributos_carrera = ["race_name", "date"]

# Generar todas las combinaciones posibles de la clave (excepto la completa)
subsets = []
for i in range(1, len(clave_carrera)):
    subsets.extend(list(combinations(clave_carrera, i)))

# Analizar dependencias
for subset in subsets:
    
    for attr in atributos_carrera:
        check = df.groupby(list(subset))[attr].nunique()
        violations = check[check > 1]
        
        if len(violations) == 0:
            print(f" Posible dependencia parcial: {subset} -> {attr}")

 Posible dependencia parcial: ('year', 'sequence') -> race_name
 Posible dependencia parcial: ('year', 'sequence') -> date


Después del análisis nos damos cuenta de una posible dependencia parcial del subconjunto (year, secuence), por tanto verificamos si este subconjunto puede determinar el atributo category.

In [128]:
check = df.groupby(["year", "sequence"])["category"].nunique()
print(check[check > 1])

year  sequence
2000  1           3
      2           3
      3           3
      4           2
      5           3
                 ..
2021  12          3
      13          3
      14          4
      15          3
      16          3
Name: category, Length: 369, dtype: int64


Se observa que (year, sequence) no determina category, pero sí determina atributos como race_name o date . Esto implica la existencia de dependencias parciales respecto a la clave compuesta (year, sequence, category), por lo que la relación no cumple la Segunda Forma Normal y debe descomponerse.

```plaintext
GRAN_PREMIO (año, secuencia, nombre, fecha, id_circuito)
                 \___ PK ___/                  ↑FK

CARRERA     (año, secuencia, categoria)
                 \_______ PK _______/
                 ↑FK→GRAN_PREMIO
```

GRAN_PREMIO recoge la información del evento físico (único por fin de semana), mientras que CARRERA recoge cada prueba dentro de ese evento por categoría.

#### ``RESULTADO``

##### 1. Comprobar la Dependencia Completa

Primero verificamos que la clave compuesta determina unívocamente a los atributos de RESULTADO:

Como en el dataset no existe ``id_piloto`` todavía (es una clave artificial que generaremos en SQL), usamos ``rider_first_name`` + ``rider_last_name`` como proxy del piloto:

In [129]:
# Clave compuesta de RESULTADO
clave_resultado = ["year", "sequence", "category",
                   "rider_first_name", "rider_last_name"]

# Atributos propios de RESULTADO 
atributos_resultado = ["position","points","speed","time","bike", "rider_number"]

print("=== Comprobación de Dependencia Funcional ===\n")
for attr in atributos_resultado:
    # Agrupamos por la clave compuesta y contamos los valores únicos del atributo
    max_valores = df.groupby(clave_resultado)[attr].nunique().max()
    
    if max_valores == 1:
        print(f"- {attr}: Depende funcionalmente de la clave (máx {max_valores} valor por grupo).")
    else:
        print(f"- {attr}: NO depende funcionalmente (hasta {max_valores} valores distintos por grupo).")

=== Comprobación de Dependencia Funcional ===

- position: Depende funcionalmente de la clave (máx 1 valor por grupo).
- points: Depende funcionalmente de la clave (máx 1 valor por grupo).
- speed: Depende funcionalmente de la clave (máx 1 valor por grupo).
- time: Depende funcionalmente de la clave (máx 1 valor por grupo).
- bike: Depende funcionalmente de la clave (máx 1 valor por grupo).
- rider_number: Depende funcionalmente de la clave (máx 1 valor por grupo).


##### 2. Comprobar Dependencias Parciales

Para que esté en 2FN, los atributos no solo deben depender de la clave completa, sino que no deben depender de un subconjunto de ella.

In [130]:
# Generar todas las combinaciones posibles de la clave (excepto la completa)
subsets = []
for i in range(1, len(clave_resultado)):
    subsets.extend(list(combinations(clave_resultado, i)))

# Analizar dependencias
for subset in subsets:
    
    for attr in atributos_resultado:
        check = df.groupby(list(subset))[attr].nunique()
        violations = check[check > 1]
        
        if len(violations) == 0:
            print(f" Posible dependencia parcial: {subset} -> {attr}")

 Posible dependencia parcial: ('year', 'sequence', 'rider_first_name', 'rider_last_name') -> position
 Posible dependencia parcial: ('year', 'sequence', 'rider_first_name', 'rider_last_name') -> points
 Posible dependencia parcial: ('year', 'sequence', 'rider_first_name', 'rider_last_name') -> speed
 Posible dependencia parcial: ('year', 'sequence', 'rider_first_name', 'rider_last_name') -> time
 Posible dependencia parcial: ('year', 'sequence', 'rider_first_name', 'rider_last_name') -> bike
 Posible dependencia parcial: ('year', 'sequence', 'rider_first_name', 'rider_last_name') -> rider_number


Tras analizar ambas entidades (CARRERA y RESULTADO), nos damos cuenta que ambos resultados apuntan a la misma causa raíz: en MotoGP, ``category`` no describe cuándo ni dónde ocurre un evento (eso lo determinan ``year`` y ``sequence``) sino qué tipo de prueba se disputa dentro de ese evento. Son dos dimensiones conceptualmente distintas que el modelo original comprimía en una sola entidad.

Esto confirma empíricamente la separación en dos tablas que propusimos en el análisis de la 2FN de CARRERA:

```plaintext
GRAN_PREMIO (año, secuencia, race_name, fecha, id_circuito)
            \____PK_____/
            

CARRERA     (año, secuencia, categoria)
            \_________PK__________/
            La prueba dentro del evento: MotoGP / Moto2 / Moto3
```

- GRAN_PREMIO agrupa lo que es común a todas las categorías de un mismo fin de semana: nombre, fecha y circuito.

- CARRERA individualiza cada prueba competitiva dentro de ese evento.

Con esta separación, (year, sequence) determina correctamente race_name y date en GRAN_PREMIO sin dependencia parcial, y (year, sequence, category) es la clave completa y mínima de CARRERA. Ambas tablas cumplen la 2FN.

#### Conclusión 2FN

El análisis empírico de dependencias parciales revela que 'category' no actúa
como identificador del evento físico:
  - (year, sequence) --> race_name, date  [violación en CARRERA original]
  - (year, sequence, piloto) --> todos los atributos  [violación en RESULTADO]

Causa raíz: el modelo original fusionaba dos conceptos distintos en CARRERA:
  1. El gran premio (evento físico: circuito, fecha, nombre)
  2. La prueba competitiva (categoría disputada dentro del evento)

Solución aplicada: separación en GRAN_PREMIO(año, secuencia, ...) y
CARRERA(año, secuencia, categoria). Ambas tablas resultantes cumplen la 2FN.

### Tercera Forma Normal (3FN)

**Condición**: estar en 2FN + ningún atributo no clave depende transitivamente de otro atributo no clave


La única tabla relevante desde nuestro punto de vista es **RESULTADO**, donde hay varios atributos no clave que podrían relacionarse entre sí.



Antes de ejecutar el código, podemos plantear varios candidatos a dependencia transitiva posible:

- **``Position --> points``**: En MotoGP el sistema de puntuación asigna puntos según la posición.

- **``team_name --> bike``**: Los equipos suelen usar siempre la misma moto (Honda, Yamaha, etc). Si team_name determina bike, habría transitividad. Sin embargo, en la auditoría detectamos que un piloto puede cambiar de equipo y moto dentro de la misma temporada, por lo que esta dependencia probablemente no sea perfecta en los datos.

In [131]:
# Atributos NO clave de RESULTADO
# Nota: id_equipo (team_name) es FK, por lo tanto no pertenece a la clave primaria de RESULTADO
no_clave = ["position", "points", "speed", "time", "bike", "rider_number", "team_name"]

print("=== Comprobación de Dependencias Transitivas (3FN) ===\n")
print("Buscando si conocer el valor de un atributo A determina el valor de un atributo B...\n")

dependencias = []

for attr_a, attr_b in permutations(no_clave, 2):
    # Descartar nulos ya que introducen ruido en la validación
    subset = df[[attr_a, attr_b]].dropna()
    
    # Agrupamos por el atributo base (A) y miramos cuántos valores únicos toma (B)
    check = subset.groupby(attr_a)[attr_b].nunique()
    
    # Violaciones: Cuando un valor de A está asociado a más de 1 valor en B
    violations = check[check > 1]

    # Si NO hay violaciones (el máximo de combinaciones distintas es 1 o menos), 
    # entonces A determina a B.
    if len(violations) == 0 and len(check) > 0:
        print(f"⚠️  Posible dependencia transitiva o violación 3FN: '{attr_a}' determina unívocamente a '{attr_b}'")
        dependencias.append((attr_a, attr_b))

print("\n=== CONCLUSIÓN ===")
if not dependencias:
    print("✅ No se detectaron dependencias transitivas. RESULTADO cumple la 3FN.")
else:
    print(f"❌ RESULTADO NO cumple la 3FN debido a las {len(dependencias)} relaciones transitivas halladas.")
    print("-> Justificación conceptual: Si un atributo que no es clave principal otorga el valor directo de otro, este último debe sacarse a una tabla de dimensiones o catálogo auxiliar.")

=== Comprobación de Dependencias Transitivas (3FN) ===

Buscando si conocer el valor de un atributo A determina el valor de un atributo B...


=== CONCLUSIÓN ===
✅ No se detectaron dependencias transitivas. RESULTADO cumple la 3FN.


- **¿Por qué position no determina matemáticamente a points en este dataset?**

El sistema de puntuación no ha sido el mismo a lo largo de la historia de MotoGP.

In [132]:
print("=== Demostración: ¿Por qué Position no determina siempre Points? ===")
# Miramos cuántos puntos distintos se han otorgado al 1º, 2º y 3º puesto a lo largo del dataset
puntos_por_posicion = df[df["position"].isin([1, 2, 3])].groupby("position")["points"].unique()
print(puntos_por_posicion)


=== Demostración: ¿Por qué Position no determina siempre Points? ===
position
1    [25, 12]
2    [20, 10]
3     [16, 8]
Name: points, dtype: object


No obstante, para no asumir que las demás entidades no tienen dependencias transitivas, vamos a generar una función general para comprobar cada entidad:

In [ ]:
def verificar_3fn(nombre_tabla, columnas_no_clave, df_subset):
    """
    Verifica si hay dependencias transitivas entre atributos no clave.
    df_subset debe contener solo las columnas relevantes ya filtradas.
    """
    print(f"\n{'='*55}")
    print(f"  Tabla: {nombre_tabla}")
    print(f"  Atributos no clave: {columnas_no_clave}")
    print(f"{'='*55}")

    if len(columnas_no_clave) < 2:
        print("  ✅ Solo 1 atributo no clave --> imposible dependencia transitiva.")
        return

    encontradas = []
    for attr_a, attr_b in permutations(columnas_no_clave, 2):
        subset = df_subset[[attr_a, attr_b]].dropna()
        if subset.empty:
            continue
        check = subset.groupby(attr_a)[attr_b].nunique()
        if (check > 1).sum() == 0:
            print(f"  ⚠️  Posible transitiva: {attr_a} --> {attr_b}")
            encontradas.append((attr_a, attr_b))

    if not encontradas:
        print("  ✅ Sin dependencias transitivas. Cumple 3FN.")


# ── PILOTO ──────────────────────────────────────────────
# pais no debería determinar nombre ni apellido
piloto_cols = df[["rider_first_name","rider_last_name",
                  "rider_country"]].drop_duplicates()
verificar_3fn("PILOTO",
              ["rider_first_name", "rider_last_name", "rider_country"],
              piloto_cols)

# ── CIRCUITO ────────────────────────────────────────────
# pais no debería determinar nombre del circuito (varios circuitos por país)
circuito_cols = df[["circuit_name","circuit_country"]].drop_duplicates()
verificar_3fn("CIRCUITO",
              ["circuit_name", "circuit_country"],
              circuito_cols)

# ── GRAN_PREMIO ─────────────────────────────────────────
# race_name, date, circuit_name son atributos no clave de (year, sequence)
gp_cols = df[["year","sequence","race_name",
              "date","circuit_name"]].drop_duplicates(
              subset=["year","sequence"])
verificar_3fn("GRAN_PREMIO",
              ["race_name", "date", "circuit_name"],
              gp_cols)

# ── EQUIPO ──────────────────────────────────────────────
verificar_3fn("EQUIPO", ["team_name"], 
              df[["team_name"]].drop_duplicates())


  Tabla: PILOTO
  Atributos no clave: ['rider_first_name', 'rider_last_name', 'rider_country']
  ✅ Sin dependencias transitivas. Cumple 3FN.

  Tabla: CIRCUITO
  Atributos no clave: ['circuit_name', 'circuit_country']
  ⚠️  Posible transitiva: circuit_name → circuit_country

  Tabla: GRAN_PREMIO
  Atributos no clave: ['race_name', 'date', 'circuit_name']
  ⚠️  Posible transitiva: date → race_name
  ⚠️  Posible transitiva: date → circuit_name

  Tabla: EQUIPO
  Atributos no clave: ['team_name']
  ✅ Solo 1 atributo no clave → imposible dependencia transitiva.


1. **CIRCUITO**: ``circuit_name --> circuit_country`` 

A primera vista parece una dependencia transitiva. Sin embargo, esto no viola la 3FN si ``circuit_name`` es una **clave candidata** (es decir, identifica unívocamente cada fila de CIRCUITO). 

En la teoría de normalización, una dependencia transitiva solo es violación cuando el atributo determinante A no es clave candidata. Si cada nombre de circuito es único en el dataset, entonces circuit_name es candidato a PK alternativa y la dependencia es legítima.



In [134]:
# ¿Es circuit_name único por cada circuito?
duplicados = df['circuit_name'].value_counts()
print(duplicados[duplicados > 1].head())  
print()
# Si el mismo nombre siempre aparece con el mismo país --> es clave candidata
check = df.groupby('circuit_name')['circuit_country'].nunique()
print(check[check > 1])  # Si vacío --> circuit_name es clave candidata 

circuit_name
Circuito De Jerez                 1869
Circuit Ricardo Tormo             1810
Le Mans                           1807
Circuit De Barcelona-Catalunya    1800
Sachsenring                       1724
Name: count, dtype: Int64

Series([], Name: circuit_country, dtype: int64)


2. **GRAN_PREMIO**: ``date --> race_name`` y ``date --> circuit_name`` 

Mismo razonamiento. Pero si cada fecha corresponde a exactamente un Gran Premio (lo cual tiene sentido en MotoGP: no se celebran dos GPs el mismo día), entonces date es clave candidata de GRAN_PREMIO, y las dependencias detectadas son simplemente dos claves candidatas relacionándose entre sí — lo cual no viola la 3FN.


In [135]:
# Construir GRAN_PREMIO duplicado
gran_premio = df.groupby(['year','sequence']).agg(
    race_name=('race_name','first'),
    date=('date','first'),
    circuit_name=('circuit_name','first')
).reset_index()

# ¿Es date único por evento?
check_date = gran_premio['date'].value_counts()
print("Fechas repetidas en GRAN_PREMIO:")
print(check_date[check_date > 1])
# Si vacío --> date es clave candidata 

Fechas repetidas en GRAN_PREMIO:
Series([], Name: count, dtype: int64)


# Migración de datos

In [136]:
import sqlite3

In [137]:
# Normalizar equipos desconocidos --> NULL
df['team_name'] = df['team_name'].replace('?', None)

In [138]:
conn = sqlite3.connect('motogp.db')
conn.execute("PRAGMA foreign_keys = ON")  
cur = conn.cursor()

In [139]:
# Celda para prevenir duplicados al ejecutar varias veces el notebook
cur.execute("DROP TABLE IF EXISTS results")
cur.execute("DROP TABLE IF EXISTS races")
cur.execute("DROP TABLE IF EXISTS grand_prix")
cur.execute("DROP TABLE IF EXISTS circuits")
cur.execute("DROP TABLE IF EXISTS teams")
cur.execute("DROP TABLE IF EXISTS riders")

In [ ]:
# 1. PILOTO
pilotos = (df[['rider_first_name','rider_last_name','rider_country']]
           .drop_duplicates()
           .reset_index(drop=True))
pilotos.index += 1  # id_piloto empieza en 1
pilotos_map = {}   # (nombre, apellido) --> id_piloto

cur.execute("""
    CREATE TABLE IF NOT EXISTS riders (
        id_rider	INTEGER PRIMARY KEY AUTOINCREMENT,
        forename	TEXT	NOT NULL,
        surname		TEXT 	NOT NULL,
        nationality	TEXT	NOT NULL
    )
""")

for idx, row in pilotos.iterrows():
    cur.execute(
        "INSERT INTO riders (forename, surname, nationality) VALUES (?,?,?)",
        (row['rider_first_name'], row['rider_last_name'], row['rider_country'])
    )
    pilotos_map[(row['rider_first_name'], row['rider_last_name'])] = cur.lastrowid

print(f"✅ PILOTO: {len(pilotos_map)} registros insertados")

✅ PILOTO: 885 registros insertados


In [141]:
# 2. EQUIPO
equipos = (df[df['team_name'].notna()][['team_name']]
           .drop_duplicates()
           .reset_index(drop=True))
equipos_map = {}  # nombre --> id_equipo

cur.execute("""
    CREATE TABLE IF NOT EXISTS teams (
        id_team INTEGER PRIMARY KEY AUTOINCREMENT,
        name    TEXT NOT NULL
    )
""")

for _, row in equipos.iterrows():
    cur.execute("INSERT INTO teams (name) VALUES (?)", (row['team_name'],))
    equipos_map[row['team_name']] = cur.lastrowid

print(f"✅ EQUIPO: {len(equipos_map)} registros insertados")

✅ EQUIPO: 969 registros insertados


In [142]:
# 3. CIRCUITO
circuitos = (df[['circuit_name','circuit_country']]
             .drop_duplicates(subset=['circuit_name'])
             .reset_index(drop=True))
circuitos_map = {}  # circuit_name --> id_circuito

cur.execute("""
    CREATE TABLE IF NOT EXISTS circuits (
        id_circuit INTEGER PRIMARY KEY AUTOINCREMENT,
        name      TEXT NOT NULL,
        country   TEXT NOT NULL
    )
""")

for _, row in circuitos.iterrows():
    pais = row['circuit_country']
    cur.execute("INSERT INTO circuits (name, country) VALUES (?,?)",
                (row['circuit_name'], pais))
    circuitos_map[row['circuit_name']] = cur.lastrowid

print(f"✅ CIRCUITO: {len(circuitos_map)} registros insertados")

✅ CIRCUITO: 29 registros insertados


In [143]:
# 4. GRAN_PREMIO
cur.execute("""
    CREATE TABLE IF NOT EXISTS grand_prix (
        year        INTEGER NOT NULL,
        sequence   INTEGER NOT NULL,
        name      TEXT    NOT NULL,
        date       TEXT    NOT NULL,
        id_circuit INTEGER NOT NULL,
        PRIMARY KEY (year, sequence),
        FOREIGN KEY (id_circuit) REFERENCES circuits(id_circuit)
    )
""")

gp = (df.groupby(['year','sequence'])
        .agg(nombre=('race_name','first'),
             fecha=('date','first'),
             circuito=('circuit_name','first'))
        .reset_index())

for _, row in gp.iterrows():
    id_circ = circuitos_map[row['circuito']]
    fecha_str = str(row['fecha'])[:10] if pd.notna(row['fecha']) else None
    cur.execute(
        "INSERT OR IGNORE INTO grand_prix VALUES (?,?,?,?,?)",
        (int(row['year']), int(row['sequence']),
         row['nombre'], fecha_str, id_circ)
    )

print(f"✅ GRAN_PREMIO: {len(gp)} registros insertados")

✅ GRAN_PREMIO: 379 registros insertados


In [144]:
# 5. CARRERA
cur.execute("""
    CREATE TABLE IF NOT EXISTS races (
        year      INTEGER NOT NULL,
        sequence INTEGER NOT NULL,
        category TEXT    NOT NULL,
        PRIMARY KEY (year, sequence, category),
        FOREIGN KEY (year, sequence) REFERENCES grand_prix(year, sequence)
    )
""")

carreras = (df[['year','sequence','category']]
            .drop_duplicates()
            .reset_index(drop=True))

for _, row in carreras.iterrows():
    cur.execute("INSERT OR IGNORE INTO races VALUES (?,?,?)",
                (int(row['year']), int(row['sequence']), row['category']))

print(f"✅ CARRERA: {len(carreras)} registros insertados")

✅ CARRERA: 1121 registros insertados


In [145]:
# 6. RESULTADO
cur.execute("""
    CREATE TABLE IF NOT EXISTS results (
        id_rider        INTEGER NOT NULL,
        year            INTEGER NOT NULL,
        sequence        INTEGER NOT NULL,
        category        TEXT    NOT NULL,
        id_team         INTEGER,
        bike            TEXT,
        position        INTEGER,
        points          REAL,
        speed           REAL,
        time            TEXT,
        rider_number    INTEGER,
        PRIMARY KEY (id_rider, year, sequence, category),
        FOREIGN KEY (id_rider) REFERENCES riders(id_rider),
        FOREIGN KEY (year, sequence, category)
            REFERENCES races(year, sequence, category),
        FOREIGN KEY (id_team) REFERENCES teams(id_team)
    )
""")

errores = 0
for _, row in df.iterrows():
    id_p  = pilotos_map.get((row['rider_first_name'], row['rider_last_name']))
    id_eq = equipos_map.get(row['team_name']) if pd.notna(row['team_name']) else None

    if id_p is None:
        errores += 1
        continue

    cur.execute("""
        INSERT OR IGNORE INTO results VALUES (?,?,?,?,?,?,?,?,?,?,?)
    """, (
        id_p,
        int(row['year']), int(row['sequence']), row['category'],
        id_eq,
        row['bike']          if pd.notna(row['bike'])          else None,
        int(row['position']) if pd.notna(row['position'])      else None,
        float(row['points']) if pd.notna(row['points'])        else None,
        float(row['speed'])  if pd.notna(row['speed'])         else None,
        str(row['time'])     if pd.notna(row['time'])          else None,
        int(row['rider_number']) if pd.notna(row['rider_number']) else None
    ))

conn.commit()
print(f"✅ RESULTADO: {cur.execute('SELECT COUNT(*) FROM results').fetchone()[0]} registros insertados")
print(f"⚠️  Filas omitidas por error: {errores}")

✅ RESULTADO: 29931 registros insertados
⚠️  Filas omitidas por error: 0


## Validación post-migración

In [146]:
# ── VERIFICAR CONTEOS ─────────────────────────────────────
tablas = ['riders','teams','circuits','grand_prix','races','results']
print("\n=== Conteo final por tabla ===")
for tabla in tablas:
    n = cur.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"  {tabla:<15} → {n:>6} filas")

# ── VERIFICAR INTEGRIDAD REFERENCIAL ─────────────────────
print("\n=== Verificación de integridad referencial ===")

# ¿Algún resultado sin piloto válido?
huerfanos_piloto = cur.execute("""
    SELECT COUNT(*) FROM results r
    LEFT JOIN riders p ON r.id_rider = p.id_rider
    WHERE p.id_rider IS NULL
""").fetchone()[0]
print(f"  Resultados sin piloto válido: {huerfanos_piloto}")

# ¿Algún resultado sin carrera válida?
huerfanos_carrera = cur.execute("""
    SELECT COUNT(*) FROM results r
    LEFT JOIN races c
        ON r.year=c.year
        AND r.sequence=c.sequence
        AND r.category=c.category
    WHERE c.year IS NULL
""").fetchone()[0]
print(f"  Resultados sin carrera válida: {huerfanos_carrera}")

# ¿NULLs en id_equipo? (esperamos ~5127)
nulls_equipo = cur.execute(
    "SELECT COUNT(*) FROM results WHERE id_team IS NULL"
).fetchone()[0]
print(f"  Resultados sin equipo (NULL): {nulls_equipo}  ← esperado ~5127")

conn.close()


=== Conteo final por tabla ===
  riders          →    885 filas
  teams           →    969 filas
  circuits        →     29 filas
  grand_prix      →    379 filas
  races           →   1121 filas
  results         →  29931 filas

=== Verificación de integridad referencial ===
  Resultados sin piloto válido: 0
  Resultados sin carrera válida: 0
  Resultados sin equipo (NULL): 5127  ← esperado ~5127


## MySQL Workbench

In [147]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('motogp.db')

# Script MySQL con CREATE TABLE + INSERTs
with open('motogp_mysql.sql', 'w', encoding='utf-8') as f:

    # Cabecera
    f.write("-- Migración MotoGP: SQLite → MySQL\n")
    f.write("CREATE SCHEMA IF NOT EXISTS motogp DEFAULT CHARACTER SET utf8;\n")
    f.write("USE motogp;\n\n")
    f.write("SET FOREIGN_KEY_CHECKS = 0;\n\n")  # Desactivar FKs durante carga

    # ── DDL: CREATE TABLE ─────────────────────────────────
    f.write("""
CREATE TABLE IF NOT EXISTS riders (
    id_rider      INT             AUTO_INCREMENT,
    forename      VARCHAR(50)     NOT NULL,
    surname       VARCHAR(50)     NOT NULL,
    nationality   VARCHAR(50)     NOT NULL,
    PRIMARY KEY (id_rider)
);

CREATE TABLE IF NOT EXISTS teams (
    id_team   INT             AUTO_INCREMENT,
    name      VARCHAR(100)    NOT NULL,
    PRIMARY KEY (id_team)
);

CREATE TABLE IF NOT EXISTS circuits (
    id_circuit  INT             AUTO_INCREMENT,
    name        VARCHAR(100)    NOT NULL,
    country     VARCHAR(50)     NOT NULL,
    PRIMARY KEY (id_circuit)
);

CREATE TABLE IF NOT EXISTS grand_prix (
	year		INTEGER 		NOT NULL,
    sequence	INTEGER			NOT NULL,
    name		VARCHAR(100) 	NOT NULL,
    date		DATE			NOT NULL,
    id_circuit	INTEGER			NOT NULL,
    PRIMARY KEY	(year, sequence),
    CONSTRAINT fk_gp_circuits
		FOREIGN KEY (id_circuit)
        REFERENCES circuits (id_circuit)
        ON DELETE RESTRICT
        ON UPDATE CASCADE
);

            
CREATE TABLE IF NOT EXISTS races (
	year		INTEGER 		NOT NULL,
    sequence	INTEGER			NOT NULL,
    category	VARCHAR(20)		NOT NULL,
    PRIMARY KEY	(year, sequence, category),
    CONSTRAINT fk_race_gp
		FOREIGN KEY (year, sequence)
        REFERENCES grand_prix (year, sequence)
        ON DELETE RESTRICT
        ON UPDATE CASCADE
);

CREATE TABLE IF NOT EXISTS results (
	id_rider	INTEGER 		NOT NULL,
    year		INTEGER			NOT NULL,
    sequence	INTEGER			NOT NULL,
    category	VARCHAR(20)		NOT NULL,
    id_team		INTEGER,
    bike		VARCHAR(50),
    position	INTEGER,
    points		DECIMAL(5,1),
    speed		DECIMAL(6,3),
    time		VARCHAR(20),
    rider_number INTEGER,
    PRIMARY KEY	(id_rider, year, sequence, category),
    CONSTRAINT fk_res_rider
		FOREIGN KEY (id_rider)
        REFERENCES riders (id_rider)
        ON DELETE RESTRICT 
        ON UPDATE CASCADE,
	CONSTRAINT fk_res_race
		FOREIGN KEY (year, sequence, category)
        REFERENCES races (year, sequence, category)
        ON DELETE RESTRICT 
        ON UPDATE CASCADE,
	CONSTRAINT fk_res_team
		FOREIGN KEY (id_team)
        REFERENCES teams (id_team)
        ON DELETE SET NULL 
        ON UPDATE CASCADE
);
\n\n""")

    # ── DML: INSERT INTO ──────────────────────────────────
    tablas = ['riders', 'teams', 'circuits', 'grand_prix', 'races', 'results']

    for tabla in tablas:
        df = pd.read_sql_query(f"SELECT * FROM {tabla}", conn)
        if df.empty:
            continue

        cols = ', '.join(df.columns)
        f.write(f"-- {tabla.upper()}\n")

        # Insertar en bloques de 500 filas para mayor eficiencia
        for i in range(0, len(df), 500):
            chunk = df.iloc[i:i+500]
            values_list = []
            for _, row in chunk.iterrows():
                vals = []
                for v in row:
                    if pd.isna(v) or v is None:
                        vals.append('NULL')
                    elif isinstance(v, str):
                        # Escapar comillas simples
                        v_escaped = v.replace("'", "\\'")
                        vals.append(f"'{v_escaped}'")
                    else:
                        vals.append(str(v))
                values_list.append(f"({', '.join(vals)})")

            f.write(f"INSERT INTO {tabla} ({cols}) VALUES\n")
            f.write(',\n'.join(values_list))
            f.write(';\n')

        f.write('\n')

    # Reactivar FKs al final
    f.write("SET FOREIGN_KEY_CHECKS = 1;\n")

    # ── ACTUALIZAR CÓDIGOS DE PAÍSES ──────────────────────
    f.write("-- -- Modificar los valores del atributo country para que coincidan con el atributo nationality y poder realizar la consulta 6\n")
    f.write("""UPDATE circuits
SET country = CASE country
    WHEN 'AR' THEN 'ARG'
    WHEN 'AT' THEN 'AUT'
    WHEN 'AU' THEN 'AUS'
    WHEN 'BR' THEN 'BRA'
    WHEN 'CN' THEN 'CHN'
    WHEN 'CZ' THEN 'CZE'
    WHEN 'DE' THEN 'GER'
    WHEN 'ES' THEN 'SPA'
    WHEN 'FR' THEN 'FRA'
    WHEN 'GB' THEN 'GBR'
    WHEN 'IT' THEN 'ITA'
    WHEN 'JP' THEN 'JPN'
    WHEN 'MY' THEN 'MAL'
    WHEN 'NL' THEN 'NED'
    WHEN 'PT' THEN 'POR'
    WHEN 'QA' THEN 'QAT'
    WHEN 'TH' THEN 'THA'
    WHEN 'TR' THEN 'TUR'
    WHEN 'US' THEN 'USA'
    WHEN 'ZA' THEN 'RSA'
    ELSE country
END
WHERE id_circuit >= 1;
\n""")

conn.close()
print("✅ Archivo 'motogp_mysql.sql' generado correctamente")

✅ Archivo 'motogp_mysql.sql' generado correctamente
